**Monter Google colab+charger best.pt**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
model = YOLO(
    "/content/drive/MyDrive/YOLO_Results/license_plate_characters-3/weights/best.pt"
)

print("✅ Modèle chargé")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Modèle chargé


**Tester sur une image de plaque**

In [ ]:
image_path = "/content/drive/MyDrive/LicensePlate_YOLO/test/images/1096_jpg.rf.b62dd05f867d4a607831e3306c20ca58_plate_0.jpg"

In [ ]:
results = model.predict(
    source=image_path,
    conf=0.25,
    save=True
)


image 1/1 /content/drive/MyDrive/LicensePlate_YOLO/test/images/1096_jpg.rf.b62dd05f867d4a607831e3306c20ca58_plate_0.jpg: 192x640 1 TN, 2 0s, 1 1, 3 7s, 6.1ms
Speed: 72.1ms preprocess, 6.1ms inference, 39.4ms postprocess per image at shape (1, 3, 192, 640)
Results saved to /content/runs/detect/predict


**Afficher ce que YOLO détecte --> les classes**

In [ ]:
#Avant de reconstruire le matricule, on vérifie les sorties :

for r in results:
    print("Image :", r.path)

    for box in r.boxes:
        class_id = int(box.cls)
        confidence = float(box.conf)

        print(
            "Classe :", model.names[class_id],
            "| confiance :", round(confidence,3),
            "| bbox :", box.xyxy.tolist()
        )

Image : /content/drive/MyDrive/LicensePlate_YOLO/test/images/1096_jpg.rf.b62dd05f867d4a607831e3306c20ca58_plate_0.jpg
Classe : 0 | confiance : 0.971 | bbox : [[122.91973876953125, 9.60088062286377, 136.1397705078125, 30.69501495361328]]
Classe : 0 | confiance : 0.96 | bbox : [[60.803619384765625, 11.046178817749023, 73.5225830078125, 33.39105987548828]]
Classe : TN | confiance : 0.931 | bbox : [[74.72559356689453, 10.610404968261719, 110.15214538574219, 31.89826011657715]]
Classe : 7 | confiance : 0.895 | bbox : [[151.04336547851562, 9.771580696105957, 163.88577270507812, 31.25699806213379]]
Classe : 7 | confiance : 0.855 | bbox : [[46.5078239440918, 11.255721092224121, 59.12611389160156, 33.159080505371094]]
Classe : 7 | confiance : 0.832 | bbox : [[136.8011932373047, 9.879670143127441, 149.89817810058594, 31.424901962280273]]
Classe : 1 | confiance : 0.614 | bbox : [[37.273075103759766, 11.280261039733887, 44.4060173034668, 32.14973068237305]]


**Construire la fonction de reconnaissance**

In [ ]:
def recognize_plate(result, model):
    characters = []

    for box in result.boxes:
        class_id = int(box.cls)# box.cls est un tensor PyTorch, donc on le convertit en entier+ on recupere l'id de la classe predite:Exple : 0 = TN, 1 = '0', 2 = '1',..., 10 = '9'.
        label = model.names[class_id] # associer l'id de la classe à son nom réel grâce au dictionnair "model.names" Exple : si class_id = 6, alors label = '5'.

        x1, y1, x2, y2 = box.xyxy[0].tolist() #coordonnées de la bounding box

        center_x = (x1 + x2) / 2 # cette valeur permettra de trier les caractères de gauche à droite

        characters.append(  #ajouter le tuple (centre horizontal, caractère détecté) :Exple  (45.8, 'TN') / (96.2, '3') / (135.7, '8')
            (center_x, label)
        )

    # tri gauche → droite
    characters.sort(key=lambda x: x[0]) #[(135, '8'),               # [(25, 'TN'),
                                        #  (25, 'TN'),     -------> # (65, '5'),
                                        #  (65, '5'),               # (100, '3'),
                                        #  (100, '3') ]             # (135, '8') ]

    plate_text = ""

    for _, char in characters:
        plate_text += char

    return plate_text

In [ ]:
#obtenir la matricule
plate = recognize_plate(results[0], model)
print("Matricule détecté :", plate)

Matricule détecté : 170TN077
